# 04 — PyTorch Neural Network Fundamentals

A neural network is not a black box: tensors flow through layers, the forward pass produces logits, a loss measures error, backpropagation calculates gradients and an optimiser updates weights. This notebook keeps the dataset synthetic so the mechanics stay visible.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
n = 1200
X = torch.randn(n, 2)
# Non-linear target: points outside a radius are class 1.
y = ((X[:, 0] ** 2 + X[:, 1] ** 2) > 1.0).float().unsqueeze(1)

indices = torch.randperm(n)
split = int(0.8 * n)
train_idx, test_idx = indices[:split], indices[split:]
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print(X_train.shape, y_train.shape)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.network(x)

model = MLP()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model

In [ ]:
for epoch in range(201):
    model.train()
    optimizer.zero_grad()                 # clear old gradients
    logits = model(X_train)               # forward pass
    loss = loss_fn(logits, y_train)       # measure error
    loss.backward()                       # backpropagation
    optimizer.step()                      # update weights

    if epoch % 50 == 0:
        print(f'epoch={epoch:3d} loss={loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_test))
    predictions = (probabilities >= 0.5).float()
    accuracy = (predictions == y_test).float().mean().item()
print(f'test accuracy: {accuracy:.3f}')

## What each part means

- `nn.Linear`: learns weights and biases.
- `ReLU`: introduces non-linearity; without activations, stacked linear layers are still just a linear function.
- `BCEWithLogitsLoss`: combines a numerically stable sigmoid operation with binary cross-entropy.
- `loss.backward()`: automatic differentiation computes gradients for every trainable parameter.
- `optimizer.step()`: changes the parameters using those gradients.
- `model.eval()` + `torch.no_grad()`: evaluation mode without building a gradient graph.